# LOB Time Series Analysis

This notebook is focused on one task only: for a chosen `token_id`, build a clean 5-level limit order book time series and analyze whether the book changed over the sampled period.

The notebook intentionally avoids duplicated top-of-book analysis. Instead, it creates one canonical representation that is suitable for future custom LOB feature extraction.

## 1. Open The Database

The notebook reads the SQLite database produced by the polling collector. Processed artifacts are written under `../cached_cached_data/processed`.

In [12]:
from pathlib import Path
import sqlite3

from IPython.display import display
import pandas as pd

# DB_PATH = Path("../db/iran_conflict_orderbooks.sqlite")
DB_PATH = Path("../db/us_forces.sqlite")
PROCESSED_DIR = Path("../cached_cached_data/processed")
MAX_LEVELS = 5

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 300)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", str)
pd.set_option("display.colheader_justify", "left")
pd.set_option("display.max_colwidth", 200)

print(DB_PATH.resolve())


/Users/sneddy/research/polymarket_research/db/us_forces.sqlite


## 2. Helper Functions

The helpers below do three things:
- resolve the token and inspect its metadata
- load the saved 5-level bid/ask history for that token
- build a canonical wide LOB panel and a change-detection layer on top of it

This canonical panel is the intended input to a future customizable LOB feature extractor.

In [13]:
def query_df(sql: str, params: tuple = ()) -> pd.DataFrame:
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn, params=params)


def list_tokens() -> pd.DataFrame:
    return query_df(
        """
        SELECT
            source_slug,
            market_slug,
            outcome_name,
            token_id,
            condition_id,
            market_question,
            group_item_title
        FROM market_outcomes
        ORDER BY market_slug, outcome_index
        """
    )


def find_tokens(market_slug: str | None = None, outcome_name: str | None = None) -> pd.DataFrame:
    return query_df(
        """
        SELECT
            source_slug,
            market_slug,
            outcome_name,
            token_id,
            condition_id,
            market_question,
            group_item_title
        FROM market_outcomes
        WHERE (? IS NULL OR market_slug = ?)
          AND (? IS NULL OR outcome_name = ?)
        ORDER BY market_slug, outcome_index
        """,
        (market_slug, market_slug, outcome_name, outcome_name),
    )


def resolve_token_id(
    token_id: str | None = None,
    market_slug: str | None = None,
    outcome_name: str | None = None,
) -> str:
    if token_id is not None:
        return str(token_id)

    candidates = find_tokens(market_slug=market_slug, outcome_name=outcome_name)
    if candidates.empty:
        raise ValueError("No token matched the provided market_slug/outcome_name filter")
    if len(candidates) != 1:
        raise ValueError(
            "Filter is not unique. Narrow it until exactly one token remains. "
            f"Matched rows: {len(candidates)}"
        )
    return str(candidates.iloc[0]["token_id"])


def load_token_metadata(token_id: str) -> pd.DataFrame:
    return query_df(
        """
        SELECT
            source_slug,
            market_slug,
            outcome_name,
            token_id,
            condition_id,
            market_question,
            group_item_title,
            active,
            closed,
            archived,
            enable_order_book,
            updated_at_utc
        FROM market_outcomes
        WHERE token_id = ?
        """,
        (token_id,),
    )


def load_token_levels_ts(token_id: str, max_levels: int = 5) -> pd.DataFrame:
    return query_df(
        """
        SELECT
            s.captured_at_utc,
            s.market_slug,
            s.outcome_name,
            s.token_id,
            s.condition_id,
            s.book_timestamp_ms,
            l.side,
            l.level_index,
            l.price,
            l.size,
            l.price * l.size AS notional
        FROM orderbook_snapshots s
        JOIN orderbook_levels l ON l.snapshot_id = s.id
        WHERE s.token_id = ?
          AND l.level_index < ?
        ORDER BY s.captured_at_utc, l.side, l.level_index
        """,
        (token_id, int(max_levels)),
    )


def build_lob_panel(levels_df: pd.DataFrame, max_levels: int = 5) -> pd.DataFrame:
    if levels_df.empty:
        return levels_df.copy()

    panel = (
        levels_df.assign(level_key=lambda df: df["side"] + "_" + df["level_index"].astype(str))
        .pivot_table(
            index=["captured_at_utc", "market_slug", "outcome_name", "token_id", "condition_id", "book_timestamp_ms"],
            columns="level_key",
            values=["price", "size"],
            aggfunc="first",
        )
    )

    panel.columns = [f"{value}_{level}" for value, level in panel.columns]
    panel = panel.reset_index().sort_values("captured_at_utc").reset_index(drop=True)

    expected_cols = []
    for value_name in ["price", "size"]:
        for side_name in ["bid", "ask"]:
            for level_idx in range(int(max_levels)):
                expected_cols.append(f"{value_name}_{side_name}_{level_idx}")

    for col in expected_cols:
        if col not in panel.columns:
            panel[col] = pd.NA

    panel["observed_bid_levels"] = panel[[f"price_bid_{i}" for i in range(int(max_levels))]].notna().sum(axis=1)
    panel["observed_ask_levels"] = panel[[f"price_ask_{i}" for i in range(int(max_levels))]].notna().sum(axis=1)

    ordered_cols = [
        "captured_at_utc",
        "market_slug",
        "outcome_name",
        "token_id",
        "condition_id",
        "book_timestamp_ms",
        "observed_bid_levels",
        "observed_ask_levels",
    ] + expected_cols
    return panel[ordered_cols]


def build_book_change_df(lob_panel_df: pd.DataFrame, max_levels: int = 5) -> pd.DataFrame:
    if lob_panel_df.empty:
        return lob_panel_df.copy()

    out = lob_panel_df.copy().sort_values("captured_at_utc").reset_index(drop=True)
    out["captured_at_utc"] = pd.to_datetime(out["captured_at_utc"], utc=True)
    out["prev_captured_at_utc"] = out["captured_at_utc"].shift(1)
    out["seconds_since_prev"] = (out["captured_at_utc"] - out["prev_captured_at_utc"]).dt.total_seconds()
    out["has_prev_snapshot"] = out["prev_captured_at_utc"].notna()

    bid_cols = [f"{value}_{side}_{i}" for value in ["price", "size"] for side in ["bid"] for i in range(int(max_levels))]
    ask_cols = [f"{value}_{side}_{i}" for value in ["price", "size"] for side in ["ask"] for i in range(int(max_levels))]
    book_cols = bid_cols + ask_cols

    prev = out[book_cols].shift(1)
    changed = out[book_cols].ne(prev) & ~(out[book_cols].isna() & prev.isna())

    out["book_changed_any"] = changed.any(axis=1) & out["has_prev_snapshot"]
    out["bid_changed_any"] = changed[bid_cols].any(axis=1) & out["has_prev_snapshot"]
    out["ask_changed_any"] = changed[ask_cols].any(axis=1) & out["has_prev_snapshot"]
    out["changed_field_count"] = changed.sum(axis=1).where(out["has_prev_snapshot"], 0)
    out["best_bid_price_changed"] = changed["price_bid_0"].where(out["has_prev_snapshot"], False)
    out["best_ask_price_changed"] = changed["price_ask_0"].where(out["has_prev_snapshot"], False)
    out["best_bid_size_changed"] = changed["size_bid_0"].where(out["has_prev_snapshot"], False)
    out["best_ask_size_changed"] = changed["size_ask_0"].where(out["has_prev_snapshot"], False)

    return out


def summarize_book_changes(change_df: pd.DataFrame) -> pd.DataFrame:
    if change_df.empty:
        return change_df.copy()

    comparable = change_df[change_df["has_prev_snapshot"]].copy()
    if comparable.empty:
        return pd.DataFrame([
            {
                "snapshots": len(change_df),
                "comparable_transitions": 0,
                "changed_transitions": 0,
                "unchanged_transitions": 0,
                "change_rate": pd.NA,
                "avg_changed_field_count": pd.NA,
            }
        ])

    return pd.DataFrame([
        {
            "snapshots": len(change_df),
            "comparable_transitions": len(comparable),
            "changed_transitions": int(comparable["book_changed_any"].sum()),
            "unchanged_transitions": int((~comparable["book_changed_any"]).sum()),
            "change_rate": float(comparable["book_changed_any"].mean()),
            "avg_changed_field_count": float(comparable["changed_field_count"].mean()),
        }
    ])


def export_df(df: pd.DataFrame, path: str | Path) -> Path:
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    if out.suffix == ".csv":
        df.to_csv(out, index=False)
    elif out.suffix == ".parquet":
        df.to_parquet(out, index=False)
    else:
        raise ValueError("Supported suffixes: .csv, .parquet")
    return out


## 3. Inspect Available Tokens

Use this table only if you need to discover or verify the token. If `TOKEN_ID` is already known, you can skip directly to the next section.

In [14]:
tokens_df = list_tokens()
display(tokens_df)


## 4. Choose The Token

The notebook is designed to work from a specific token. You can either set `TOKEN_ID` directly or resolve it through a unique `MARKET_SLUG` and `OUTCOME_NAME` pair.

In [15]:
# TOKEN_ID = '42541597022580268380579424081239402820400625604765321493406160729090848893639' #15 april
TOKEN_ID = '42750054381142639205639663180818682570869285140532640407891991570656047928885' #us forces

MARKET_SLUG = None
OUTCOME_NAME = None

resolved_token_id = resolve_token_id(
    token_id=TOKEN_ID,
    market_slug=MARKET_SLUG,
    outcome_name=OUTCOME_NAME,
)

token_meta_df = load_token_metadata(resolved_token_id)
display(token_meta_df)
print('resolved_token_id =', resolved_token_id)


resolved_token_id = 42750054381142639205639663180818682570869285140532640407891991570656047928885


## 5. Load The Saved 5-Level Book History

This is the raw time series of saved bid and ask levels for the chosen token. The query below is explicitly capped at the first 5 levels on each side.

In [16]:
levels_ts_df = load_token_levels_ts(resolved_token_id, max_levels=MAX_LEVELS)
display(levels_ts_df)


## 6. Build The Canonical 5-Level LOB Panel

This panel is the core representation for downstream work. It stores one row per timestamp and separate columns for:
- `price_bid_0` ... `price_bid_4`
- `size_bid_0` ... `size_bid_4`
- `price_ask_0` ... `price_ask_4`
- `size_ask_0` ... `size_ask_4`

This is the right level of representation for future custom feature extraction.

In [17]:
lob_panel_df = build_lob_panel(levels_ts_df, max_levels=MAX_LEVELS)
display(lob_panel_df)


## 7. Detect Whether The Book Changed Over Time

This section compares each snapshot to the immediately previous snapshot for the same token.

The resulting table answers questions such as:
- did anything in the saved 5-level book change?
- did only the bid side change?
- did only the ask side change?
- how many fields changed between snapshots?
- did the best bid or best ask move?


In [18]:
usecols = [
    'captured_at_utc', 'book_changed_any',
    'bid_changed_any', 'ask_changed_any',
    'size_bid_0', 'size_bid_1', 'size_bid_2', 'size_bid_3', 'size_bid_4',
    'size_ask_0', 'size_ask_1', 'size_ask_2', 'size_ask_3', 'size_ask_4',
    # 'changed_field_count',
    # 'best_bid_price_changed', 'best_ask_price_changed',
    # 'best_bid_size_changed', 'best_ask_size_changed'
]
book_change_df = build_book_change_df(lob_panel_df, max_levels=MAX_LEVELS)
display(book_change_df[usecols])
book_change_df.book_changed_any.mean()


## 8. Summarize Change Frequency

This compact summary tells you whether the sampled period was mostly static or whether the saved 5-level book was changing often.

In [19]:
book_change_summary_df = summarize_book_changes(book_change_df)
display(book_change_summary_df)


## 9. Build A Simple LOB Feature Extractor

Instead of exporting raw artifacts, use the canonical 5-level panel to build a compact feature table.

The extractor below focuses on common LOB features:
- best bid / best ask / spread / relative spread / mid price
- microprice and top-level imbalance
- cumulative depth and depth imbalance over 1, 3, and 5 levels
- depth-weighted prices over 1, 3, and 5 levels
- one-step quote changes and simple mid-price returns
- change flags derived from the saved book snapshots

This keeps the notebook simple while already giving a useful baseline feature set.

In [20]:
def _safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    denom = denominator.replace(0, pd.NA)
    return numerator / denom


def build_lob_features(
    lob_panel: pd.DataFrame,
    book_change: pd.DataFrame | None = None,
    max_levels: int = 5,
    depth_levels: tuple[int, ...] = (1, 3, 5),
) -> pd.DataFrame:
    if lob_panel.empty:
        return lob_panel.copy()

    df = lob_panel.copy().sort_values('captured_at_utc').reset_index(drop=True)
    df['captured_at_utc'] = pd.to_datetime(df['captured_at_utc'], utc=True)
    df['seconds_since_prev'] = df['captured_at_utc'].diff().dt.total_seconds()

    df['best_bid'] = df['price_bid_0']
    df['best_ask'] = df['price_ask_0']
    df['best_bid_size'] = df['size_bid_0']
    df['best_ask_size'] = df['size_ask_0']
    df['spread'] = df['best_ask'] - df['best_bid']
    df['mid_price'] = (df['best_bid'] + df['best_ask']) / 2.0
    df['relative_spread'] = _safe_divide(df['spread'], df['mid_price'])
    df['top_level_imbalance'] = _safe_divide(
        df['best_bid_size'] - df['best_ask_size'],
        df['best_bid_size'] + df['best_ask_size'],
    )
    df['microprice'] = _safe_divide(
        df['best_ask'] * df['best_bid_size'] + df['best_bid'] * df['best_ask_size'],
        df['best_bid_size'] + df['best_ask_size'],
    )

    if 'price_bid_1' in df.columns:
        df['bid_gap_0_1'] = df['price_bid_0'] - df['price_bid_1']
    if 'price_ask_1' in df.columns:
        df['ask_gap_0_1'] = df['price_ask_1'] - df['price_ask_0']

    for depth in depth_levels:
        if depth > int(max_levels):
            continue

        bid_size_cols = [f'size_bid_{i}' for i in range(depth)]
        ask_size_cols = [f'size_ask_{i}' for i in range(depth)]
        bid_price_cols = [f'price_bid_{i}' for i in range(depth)]
        ask_price_cols = [f'price_ask_{i}' for i in range(depth)]

        bid_depth = df[bid_size_cols].sum(axis=1, min_count=1)
        ask_depth = df[ask_size_cols].sum(axis=1, min_count=1)
        bid_notional = sum(df[p] * df[s] for p, s in zip(bid_price_cols, bid_size_cols))
        ask_notional = sum(df[p] * df[s] for p, s in zip(ask_price_cols, ask_size_cols))

        df[f'bid_depth_{depth}'] = bid_depth
        df[f'ask_depth_{depth}'] = ask_depth
        df[f'depth_imbalance_{depth}'] = _safe_divide(bid_depth - ask_depth, bid_depth + ask_depth)
        df[f'bid_wap_{depth}'] = _safe_divide(bid_notional, bid_depth)
        df[f'ask_wap_{depth}'] = _safe_divide(ask_notional, ask_depth)

    df['delta_best_bid'] = df['best_bid'].diff()
    df['delta_best_ask'] = df['best_ask'].diff()
    df['delta_spread'] = df['spread'].diff()
    df['mid_return_1'] = df['mid_price'].pct_change()
    df['microprice_return_1'] = df['microprice'].pct_change()

    if book_change is not None and not book_change.empty:
        change_cols = [
            'captured_at_utc',
            'book_changed_any',
            'bid_changed_any',
            'ask_changed_any',
            'changed_field_count',
            'best_bid_price_changed',
            'best_ask_price_changed',
            'best_bid_size_changed',
            'best_ask_size_changed',
        ]
        change_view = book_change[change_cols].copy()
        change_view['captured_at_utc'] = pd.to_datetime(change_view['captured_at_utc'], utc=True)
        df = df.merge(change_view, on='captured_at_utc', how='left')

    return df


lob_features_df = build_lob_features(
    lob_panel=lob_panel_df,
    book_change=book_change_df,
    max_levels=MAX_LEVELS,
    depth_levels=(1, 3, 5),
)

feature_view_cols = [
    'captured_at_utc',
    'best_bid', 'best_ask', 'spread', 'mid_price', 'microprice',
    'top_level_imbalance',
    'bid_depth_1', 'ask_depth_1', 'depth_imbalance_1',
    'bid_depth_3', 'ask_depth_3', 'depth_imbalance_3',
    'bid_depth_5', 'ask_depth_5', 'depth_imbalance_5',
    'delta_best_bid', 'delta_best_ask', 'delta_spread',
    'mid_return_1', 'microprice_return_1',
    'book_changed_any', 'changed_field_count',
]

display(lob_features_df[feature_view_cols])


In [21]:
lob_features_df.loc[:, 'captured_at_utc'] = pd.to_datetime(lob_features_df['captured_at_utc']).dt.floor('s')
lob_features_df.set_index('captured_at_utc').mid_price.plot()

In [26]:
lob_features_df.mid_price.value_counts()

In [27]:
lob_features_df.spread.value_counts()

## 10. Research Candidate Targets

The next step is target design rather than modeling.

For Polymarket-style LOB data, useful targets often look like:
- will a meaningful repricing happen within the next `h` hours?
- how long does it take until the next meaningful repricing?

The cells below build these target candidates directly from the saved `mid_price` path.

In [14]:
def infer_tick_size(lob_features: pd.DataFrame) -> float:
    price_cols = [col for col in lob_features.columns if col.startswith('price_bid_') or col.startswith('price_ask_')]
    stacked = pd.concat([lob_features[col] for col in price_cols], axis=0)
    values = pd.Series(stacked).dropna().astype(float).unique()
    values = sorted(values)
    if len(values) < 2:
        return 0.01

    diffs = pd.Series(values).diff().dropna()
    diffs = diffs[diffs > 0]
    if diffs.empty:
        return 0.01
    return float(diffs.min())


def build_repricing_targets(
    lob_features: pd.DataFrame,
    horizons_hours: tuple[int, ...] = (1, 2, 4, 6),
    move_in_ticks: int = 2,
    tick_size: float | None = None,
) -> pd.DataFrame:
    if lob_features.empty:
        return lob_features.copy()

    df = lob_features.copy().sort_values('captured_at_utc').reset_index(drop=True)
    df['captured_at_utc'] = pd.to_datetime(df['captured_at_utc'], utc=True)
    effective_tick = float(tick_size) if tick_size is not None else infer_tick_size(df)
    threshold = float(move_in_ticks) * effective_tick

    df['effective_tick_size'] = effective_tick
    df['repricing_threshold'] = threshold

    timestamps = df['captured_at_utc']
    mid_prices = pd.to_numeric(df['mid_price'], errors='coerce')

    time_to_repricing_seconds = []
    repriced_future = []

    for i in range(len(df)):
        current_t = timestamps.iloc[i]
        current_mid = mid_prices.iloc[i]
        future = df.iloc[i + 1 :].copy()

        if pd.isna(current_mid) or future.empty:
            time_to_repricing_seconds.append(pd.NA)
            repriced_future.append(False)
            continue

        future_mid = pd.to_numeric(future['mid_price'], errors='coerce')
        future_diff = (future_mid - current_mid).abs()
        hits = future.loc[future_diff >= threshold, 'captured_at_utc']

        if hits.empty:
            time_to_repricing_seconds.append(pd.NA)
            repriced_future.append(False)
        else:
            first_hit = pd.to_datetime(hits.iloc[0], utc=True)
            time_to_repricing_seconds.append((first_hit - current_t).total_seconds())
            repriced_future.append(True)

    df['repriced_future_any'] = repriced_future
    df['time_to_repricing_seconds'] = time_to_repricing_seconds
    df['time_to_repricing_hours'] = pd.to_numeric(df['time_to_repricing_seconds'], errors='coerce') / 3600.0
    df['repricing_event_observed'] = df['time_to_repricing_seconds'].notna()

    for horizon in horizons_hours:
        label_col = f'jump_within_{horizon}h'
        df[label_col] = df['time_to_repricing_hours'].le(float(horizon)).fillna(False)

    if len(df) > 0:
        last_timestamp = timestamps.iloc[-1]
        df['remaining_history_hours'] = (last_timestamp - timestamps).dt.total_seconds() / 3600.0
        df['survival_censored'] = ~df['repricing_event_observed']
        for horizon in horizons_hours:
            eligible_col = f'jump_within_{horizon}h_eligible'
            df[eligible_col] = df['remaining_history_hours'] >= float(horizon)

    return df


def summarize_target_rates(target_df: pd.DataFrame, horizons_hours: tuple[int, ...] = (1, 2, 4, 6)) -> pd.DataFrame:
    rows = []
    for horizon in horizons_hours:
        label_col = f'jump_within_{horizon}h'
        eligible_col = f'jump_within_{horizon}h_eligible'
        eligible = target_df[target_df[eligible_col]].copy() if eligible_col in target_df.columns else target_df.copy()
        rows.append({
            'horizon_hours': horizon,
            'eligible_rows': len(eligible),
            'positive_rows': int(eligible[label_col].sum()) if label_col in eligible.columns else 0,
            'positive_rate': float(eligible[label_col].mean()) if len(eligible) > 0 and label_col in eligible.columns else pd.NA,
        })
    return pd.DataFrame(rows)


MOVE_IN_TICKS = 2
TARGET_HORIZONS = (1, 2, 4, 6)

repricing_target_df = build_repricing_targets(
    lob_features=lob_features_df,
    horizons_hours=TARGET_HORIZONS,
    move_in_ticks=MOVE_IN_TICKS,
)

target_rate_df = summarize_target_rates(repricing_target_df, horizons_hours=TARGET_HORIZONS)

target_view_cols = [
    'captured_at_utc',
    'mid_price',
    'effective_tick_size',
    'repricing_threshold',
    'time_to_repricing_hours',
    'repricing_event_observed',
    'survival_censored',
] + [f'jump_within_{h}h' for h in TARGET_HORIZONS]

display(repricing_target_df[target_view_cols])
display(target_rate_df)


## 11. Interpret The Candidate Targets

Use the outputs above to answer three questions before modeling:
- is the target too rare to model reliably?
- does the positive rate become reasonable only at longer horizons such as 4h or 6h?
- is time-to-repricing mostly censored, suggesting a survival-style setup may be more natural than fixed-horizon classification?

If short horizons are almost always negative and censoring is heavy, `time-to-repricing` may be the cleaner target. If one of the fixed horizons has a workable event rate, then `jump_within_h` is a good classification baseline.

In [15]:
repricing_target_df[['captured_at_utc', 'mid_price', 'time_to_repricing_hours']].set_index('captured_at_utc').plot(
    subplots=True,
    figsize=(12, 6),
    title=['Mid Price', 'Time To Repricing (hours)'],
    legend=False,
);


## 12. Sensible Targets When Mid Price Is Sticky

If `mid_price` spends most of its time at one or two discrete values, level prediction is usually not a good research target. In that regime, more meaningful targets are event-based or state-based.

The most useful target families are:
- `repricing_within_h`: a jump-style event target over horizons such as 1h, 2h, 4h, or 6h
- `time_to_repricing`: a survival-style target for waiting time until the next meaningful repricing
- `book_activity`: whether the saved book changes materially in the next window, even if price does not
- `regime_shift`: whether the market transitions from a stale book to an active book

In practice, this means the right prediction problem is often not "what is the next mid price?" but rather "will the state of the market change in a meaningful way within a useful horizon?"

In [33]:
target_design_df = pd.DataFrame([
    {
        'target_family': 'repricing_within_h',
        'example_label': 'jump_within_4h',
        'good_when': 'Price is sticky but meaningful repricing occasionally happens.',
        'notes': 'Best simple baseline for classification.'
    },
    {
        'target_family': 'time_to_repricing',
        'example_label': 'time_to_repricing_hours',
        'good_when': 'Long flat periods dominate and repricing is sparse.',
        'notes': 'Natural survival-style setup with censoring.'
    },
    {
        'target_family': 'book_activity',
        'example_label': 'book_changed_any_next_window',
        'good_when': 'The book moves more often than the mid price.',
        'notes': 'Useful for state-change prediction rather than price prediction.'
    },
    {
        'target_family': 'regime_shift',
        'example_label': 'active_regime_next_1h',
        'good_when': 'You want to separate stale periods from active periods.',
        'notes': 'Often more stable than point price labels.'
    },
])

display(target_design_df)
